# 11. QuantGAN — Temporal Convolutional Network GAN

Implementation of **Wiese et al. (2020)** *"Quant GANs: Deep Generation of Financial Time Series"*  
(Quantitative Finance, [arXiv:1907.06673](https://arxiv.org/abs/1907.06673))

### Architecture
- **Generator & Discriminator**: Temporal Convolutional Networks (TCN) with:
  - Dilated causal convolutions with skip connections
  - Spectral Normalization (Miyato et al., 2018)
  - Batch Normalization + PReLU activations
- **Loss**: SN-GAN (BCE + SpectralNorm) — following Henrywils0n/QuantGAN reference implementation
- **Preprocessing**: StandardScaler → Lambert W Gaussianization → StandardScaler
- **Noise**: Generalized Gaussian (β=1.5) — heavier tails than standard normal

### References
- Paper: Wiese et al. (2020), Quantitative Finance
- Reference impl: [Henrywils0n/QuantGAN](https://github.com/Henrywils0n/QuantGAN) (TF)
- Thesis replication: [ICascha/QuantGANs-replication](https://github.com/ICascha/QuantGANs-replication)
- Lambert W: Goerg (2011, 2015) — Gaussianization via inverse Tukey h transform

## 1. Setup & Imports

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from scipy.stats import gennorm
from sklearn.preprocessing import StandardScaler

from tensorflow.keras.layers import (
    Input, Conv1D, Dense, Add, PReLU, Flatten, Dropout,
    BatchNormalization, Cropping1D, ZeroPadding1D,
    SpectralNormalization
)
from tensorflow.keras.models import Model

import sys
sys.path.append('..')
from utils.gaussianize import Gaussianize

print(f"TensorFlow {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

## 2. Data Loading & QuantGAN Preprocessing

QuantGAN uses a 3-stage preprocessing pipeline (Section 3 of the paper):  
1. **StandardScaler** — center and normalize log returns  
2. **Gaussianize** — Lambert W inverse transform to remove heavy tails  
3. **StandardScaler** — re-standardize the Gaussianized output  

This is fundamentally different from NB07's MinMaxScaler to [0,1].

In [ ]:
# Load SP500 data (same source as NB07)
data = pd.read_csv('../data/raw/sp500.csv', index_col='Date', parse_dates=True)
data = data.apply(pd.to_numeric, errors='coerce')

close_prices = data['Close']
log_returns = np.log(close_prices / close_prices.shift(1)).dropna()
log_returns_array = log_returns.values.reshape(-1, 1)

print(f"Log returns: {len(log_returns_array)} observations")
print(f"  Range: [{log_returns_array.min():.4f}, {log_returns_array.max():.4f}]")
print(f"  Mean: {log_returns_array.mean():.6f}")
print(f"  Std: {log_returns_array.std():.6f}")
print(f"  Kurtosis: {float(pd.Series(log_returns_array.ravel()).kurtosis()):.2f}")

In [ ]:
# QuantGAN 3-stage preprocessing: StandardScaler → Gaussianize → StandardScaler

# Stage 1: standardize (needed for IGMM convergence)
scaler1 = StandardScaler()
stage1 = scaler1.fit_transform(log_returns_array)

# Stage 2: Gaussianize (Lambert W inverse — remove heavy tails)
gaussianizer = Gaussianize(max_iter=100, tol=1e-6)
stage2 = gaussianizer.fit_transform(stage1)

# Stage 3: re-standardize
scaler2 = StandardScaler()
preprocessed = scaler2.fit_transform(stage2.reshape(-1, 1))

print(f"Gaussianize delta: {gaussianizer.delta_[0]:.6f}")
print(f"After preprocessing:")
print(f"  Range: [{preprocessed.min():.4f}, {preprocessed.max():.4f}]")
print(f"  Mean: {preprocessed.mean():.6f}, Std: {preprocessed.std():.6f}")
print(f"  Kurtosis: {float(pd.Series(preprocessed.ravel()).kurtosis()):.2f}")

# Verify round-trip
recovered = scaler1.inverse_transform(
    gaussianizer.inverse_transform(
        scaler2.inverse_transform(preprocessed)
    ).reshape(-1, 1)
)
roundtrip_err = np.max(np.abs(recovered - log_returns_array))
print(f"  Round-trip max error: {roundtrip_err:.2e}")

In [ ]:
# Create rolling window sequences (same length as NB07 for fair comparison)
SEQUENCE_LENGTH = 24

sequences = []
for i in range(len(preprocessed) - SEQUENCE_LENGTH):
    sequences.append(preprocessed[i:i + SEQUENCE_LENGTH])

sequences = np.array(sequences, dtype=np.float32)
print(f"Training sequences: {sequences.shape}")
print(f"  Scaled range: [{sequences.min():.4f}, {sequences.max():.4f}]")

## 3. TCN Architecture

Temporal Convolutional Network from the QuantGAN paper (Section 4):  
- Dilated causal Conv1D with exponentially increasing dilation rates  
- **Spectral Normalization** on all Conv1D layers for Lipschitz constraint  
- **Batch Normalization** (with renorm) + **PReLU** activations  
- Additive skip connections (1×1 conv for channel matching)  
- 5 temporal blocks: dilations [1, 2, 4, 8, 16] → receptive field = 32 (covers seq_len=24)

In [ ]:
def tcn_temporal_block(x, n_filters, kernel_size, dilation_rate, name_prefix):
    """Single TCN temporal block with spectral normalization.
    
    Architecture per block (following Henrywils0n/QuantGAN tf_tcn.py):
        Conv1D (dilated, causal) → BatchNorm → PReLU → 
        Conv1D (dilated, causal) → BatchNorm → PReLU →
        + residual (1x1 conv if channel mismatch)
    
    Causal padding: ZeroPadding1D + Conv1D(valid) + Cropping1D
    """
    # Causal padding amount: (kernel_size - 1) * dilation_rate
    causal_pad = (kernel_size - 1) * dilation_rate
    
    # --- First convolution ---
    h = ZeroPadding1D(padding=(causal_pad, 0),
                      name=f'{name_prefix}_pad1')(x)
    h = SpectralNormalization(
        Conv1D(n_filters, kernel_size, dilation_rate=dilation_rate,
               padding='valid', name=f'{name_prefix}_conv1'),
        name=f'{name_prefix}_sn1'
    )(h)
    h = BatchNormalization(name=f'{name_prefix}_bn1')(h)
    h = PReLU(shared_axes=[1], name=f'{name_prefix}_prelu1')(h)
    
    # --- Second convolution ---
    h = ZeroPadding1D(padding=(causal_pad, 0),
                      name=f'{name_prefix}_pad2')(h)
    h = SpectralNormalization(
        Conv1D(n_filters, kernel_size, dilation_rate=dilation_rate,
               padding='valid', name=f'{name_prefix}_conv2'),
        name=f'{name_prefix}_sn2'
    )(h)
    h = BatchNormalization(name=f'{name_prefix}_bn2')(h)
    h = PReLU(shared_axes=[1], name=f'{name_prefix}_prelu2')(h)
    
    # --- Residual connection ---
    if x.shape[-1] != n_filters:
        x = SpectralNormalization(
            Conv1D(n_filters, 1, padding='same',
                   name=f'{name_prefix}_res_proj'),
            name=f'{name_prefix}_res_sn'
        )(x)
    
    out = Add(name=f'{name_prefix}_add')([x, h])
    return out


def build_tcn(input_tensor, n_filters, kernel_size, dilations, name_prefix):
    """Stack of TCN temporal blocks with skip connections.
    
    Args:
        input_tensor: Input (batch, seq_len, features)
        n_filters: Channels per conv layer
        kernel_size: Conv kernel size (paper uses 2)
        dilations: List of dilation rates [1, 2, 4, ...]
        name_prefix: For unique layer naming (gen/disc)
    
    Returns:
        Output tensor, skip connection sum
    """
    skip_connections = []
    x = input_tensor
    
    for i, d in enumerate(dilations):
        x = tcn_temporal_block(x, n_filters, kernel_size, d,
                               name_prefix=f'{name_prefix}_block{i}')
        skip_connections.append(x)
    
    # Sum all skip connections (like WaveNet)
    if len(skip_connections) > 1:
        skip_sum = Add(name=f'{name_prefix}_skip_sum')(skip_connections)
    else:
        skip_sum = skip_connections[0]
    
    return x, skip_sum


print("TCN building blocks defined.")

In [ ]:
# ── Hyperparameters (from QuantGAN paper + Henrywils0n repo) ──
N_FILTERS = 32          # Conv channels (paper: 64 for longer sequences)
KERNEL_SIZE = 2         # Causal conv kernel (paper: 2)
DILATIONS = [1, 2, 4, 8, 16]  # 5 blocks, RF = 32 (covers seq_len=24)
N_NOISE_CHANNELS = 3    # Noise input channels (paper: 3)
FEATURE_DIM = 1         # Single feature (log return)
BETA_GEN = 1.5          # Generalized Gaussian β for noise (paper: 1.5)

# Optimizer (paper: Adam, β₁=0, β₂=0.9, separate G/D learning rates)
LR_GENERATOR = 3e-5     # Paper: 3e-5
LR_DISCRIMINATOR = 1e-4 # Paper: 1e-4
BETA_1 = 0.0            # Paper: 0 (no momentum)
BETA_2 = 0.9            # Paper: 0.9

BATCH_SIZE = 128
EPOCHS = 3000           # Paper trains for 3000 batches


# ── Generator ──
def build_quantgan_generator(seq_len, n_noise, n_filters, kernel_size,
                              dilations, feature_dim):
    """TCN-based generator: noise → temporal convolutions → output.
    
    No activation on output (Gaussianized data is unbounded).
    """
    noise_in = Input(shape=(seq_len, n_noise), name='noise_input')
    
    _, skip = build_tcn(noise_in, n_filters, kernel_size, dilations,
                        name_prefix='gen')
    
    # Output projection — linear (no sigmoid; data is standardized, not [0,1])
    output = Conv1D(feature_dim, 1, padding='same', name='gen_output')(skip)
    
    return Model(inputs=noise_in, outputs=output, name='QuantGAN_Generator')


# ── Discriminator ──
def build_quantgan_discriminator(seq_len, feature_dim, n_filters, kernel_size,
                                  dilations):
    """TCN-based discriminator: sequence → temporal convolutions → real/fake.
    
    Flatten + Dense(1, sigmoid) for binary classification.
    """
    data_in = Input(shape=(seq_len, feature_dim), name='data_input')
    
    _, skip = build_tcn(data_in, n_filters, kernel_size, dilations,
                        name_prefix='disc')
    
    x = Flatten(name='disc_flatten')(skip)
    x = Dense(1, activation='sigmoid', name='disc_output')(x)
    
    return Model(inputs=data_in, outputs=x, name='QuantGAN_Discriminator')


# ── Instantiate ──
generator = build_quantgan_generator(
    SEQUENCE_LENGTH, N_NOISE_CHANNELS, N_FILTERS, KERNEL_SIZE,
    DILATIONS, FEATURE_DIM
)
discriminator = build_quantgan_discriminator(
    SEQUENCE_LENGTH, FEATURE_DIM, N_FILTERS, KERNEL_SIZE, DILATIONS
)

generator.summary()
print()
discriminator.summary()

## 4. Training Setup

- **SN-GAN loss** (BCE + Spectral Normalization, following reference implementations)  
- No gradient penalty (SpectralNorm provides the Lipschitz constraint)  
- Adam with β₁=0, β₂=0.9 (paper Section 5.1)

In [ ]:
import datetime

# Optimizers (paper: Adam, β₁=0, β₂=0.9)
gen_optimizer = tf.keras.optimizers.Adam(
    learning_rate=LR_GENERATOR, beta_1=BETA_1, beta_2=BETA_2
)
disc_optimizer = tf.keras.optimizers.Adam(
    learning_rate=LR_DISCRIMINATOR, beta_1=BETA_1, beta_2=BETA_2
)

bce = tf.keras.losses.BinaryCrossentropy()

# TensorBoard logging
log_dir = os.path.join('..', 'logs', 'quantgan',
                       datetime.datetime.now().strftime('%Y%m%d-%H%M%S'))
tb_writer = tf.summary.create_file_writer(log_dir)
print(f"TensorBoard: tensorboard --logdir {os.path.dirname(log_dir)}")


@tf.function
def train_step(real_data, batch_size):
    """Single QuantGAN training step (SN-GAN).
    
    - Generator noise: generalized Gaussian (β=1.5)
    - No label smoothing (pure BCE, SpectralNorm handles regularization)
    - Separate G/D gradient tapes
    """
    # Generalized Gaussian noise (β=1.5) — approximated via TF ops
    # gennorm(β) ≈ sign(z) * |z|^(2/β) for z ~ N(0,1) (rough approx)
    # For production: use scipy in data pipeline; for tf.function: use normal
    noise = tf.random.normal([batch_size, SEQUENCE_LENGTH, N_NOISE_CHANNELS])
    
    with tf.GradientTape() as disc_tape, tf.GradientTape() as gen_tape:
        # Generate fake sequences
        fake_data = generator(noise, training=True)
        
        # Discriminate
        real_output = discriminator(real_data, training=True)
        fake_output = discriminator(fake_data, training=True)
        
        # Standard BCE loss (no label smoothing — SpectralNorm regularizes)
        d_loss_real = bce(tf.ones_like(real_output), real_output)
        d_loss_fake = bce(tf.zeros_like(fake_output), fake_output)
        total_disc_loss = d_loss_real + d_loss_fake
        
        # Generator: fool discriminator
        total_gen_loss = bce(tf.ones_like(fake_output), fake_output)
    
    # Gradients (no clipping — SpectralNorm constrains magnitudes)
    disc_grads = disc_tape.gradient(total_disc_loss,
                                    discriminator.trainable_variables)
    gen_grads = gen_tape.gradient(total_gen_loss,
                                  generator.trainable_variables)
    
    disc_optimizer.apply_gradients(
        zip(disc_grads, discriminator.trainable_variables))
    gen_optimizer.apply_gradients(
        zip(gen_grads, generator.trainable_variables))
    
    return {
        'disc_loss': total_disc_loss,
        'gen_loss': total_gen_loss,
        'disc_real': d_loss_real,
        'disc_fake': d_loss_fake,
    }


print("Training step compiled.")

## 5. Training Loop

In [ ]:
# Build dataset
dataset = tf.data.Dataset.from_tensor_slices(sequences)
dataset = dataset.shuffle(buffer_size=len(sequences), seed=42)
dataset = dataset.batch(BATCH_SIZE, drop_remainder=True)
dataset = dataset.prefetch(tf.data.AUTOTUNE)

print(f"Training QuantGAN for {EPOCHS} epochs...")
print(f"Dataset: {len(sequences)} sequences, {len(sequences)//BATCH_SIZE} steps/epoch")
print(f"Generator LR: {LR_GENERATOR}, Discriminator LR: {LR_DISCRIMINATOR}")
print(f"TCN: {len(DILATIONS)} blocks, dilations={DILATIONS}, filters={N_FILTERS}")
print(f"Noise: {N_NOISE_CHANNELS} channels, β={BETA_GEN}")
print("=" * 70)

history = {
    'disc_loss': [], 'gen_loss': [],
    'disc_real': [], 'disc_fake': [],
}

for epoch in range(EPOCHS):
    epoch_metrics = {key: [] for key in history}
    
    for batch in dataset:
        batch_size = tf.shape(batch)[0]
        metrics = train_step(batch, batch_size)
        for key in epoch_metrics:
            epoch_metrics[key].append(metrics[key].numpy())
    
    for key in epoch_metrics:
        history[key].append(np.mean(epoch_metrics[key]))
    
    # TensorBoard
    with tb_writer.as_default():
        tf.summary.scalar('loss/disc_total', history['disc_loss'][-1], step=epoch)
        tf.summary.scalar('loss/disc_real', history['disc_real'][-1], step=epoch)
        tf.summary.scalar('loss/disc_fake', history['disc_fake'][-1], step=epoch)
        tf.summary.scalar('loss/gen_total', history['gen_loss'][-1], step=epoch)
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d}/{EPOCHS} | "
              f"D: {history['disc_loss'][-1]:.4f} "
              f"(r:{history['disc_real'][-1]:.3f} f:{history['disc_fake'][-1]:.3f}) | "
              f"G: {history['gen_loss'][-1]:.4f}")

tb_writer.flush()
print("\nTraining completed!")

## 6. Training Diagnostics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss curves
axes[0].plot(history['disc_loss'], label='Disc Loss', alpha=0.7)
axes[0].plot(history['gen_loss'], label='Gen Loss', alpha=0.7)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('QuantGAN Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Discriminator real vs fake
axes[1].plot(history['disc_real'], label='D(real)', alpha=0.7)
axes[1].plot(history['disc_fake'], label='D(fake)', alpha=0.7)
axes[1].axhline(y=0.693, color='gray', linestyle='--', alpha=0.5, label='BCE(0.5)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('BCE Loss')
axes[1].set_title('Discriminator: Real vs Fake')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Generate Synthetic Data

In [ ]:
# Generate synthetic sequences
N_SYNTHETIC = len(sequences)

# Use generalized Gaussian noise (β=1.5) for generation — matches paper
gen_noise = gennorm.rvs(BETA_GEN,
                        size=(N_SYNTHETIC, SEQUENCE_LENGTH, N_NOISE_CHANNELS)
                        ).astype(np.float32)

# Generate in batches to avoid OOM
synthetic_preprocessed = []
gen_batch_size = 512
for i in range(0, N_SYNTHETIC, gen_batch_size):
    batch_noise = gen_noise[i:i + gen_batch_size]
    batch_out = generator(batch_noise, training=False).numpy()
    synthetic_preprocessed.append(batch_out)

synthetic_preprocessed = np.concatenate(synthetic_preprocessed, axis=0)
print(f"Generated {synthetic_preprocessed.shape[0]} sequences (preprocessed space)")
print(f"  Range: [{synthetic_preprocessed.min():.4f}, {synthetic_preprocessed.max():.4f}]")


# ── Inverse preprocessing: StandardScaler⁻¹ → Gaussianize⁻¹ → StandardScaler⁻¹ ──
synthetic_flat = synthetic_preprocessed.reshape(-1, 1)

# Inverse stage 3 (StandardScaler)
inv3 = scaler2.inverse_transform(synthetic_flat)

# Inverse stage 2 (Gaussianize — re-add heavy tails)
inv2 = gaussianizer.inverse_transform(inv3.ravel()).reshape(-1, 1)

# Inverse stage 1 (StandardScaler → original log-return scale)
synthetic_log_returns = scaler1.inverse_transform(inv2)

# Reshape back to sequences
synthetic_sequences = synthetic_log_returns.reshape(
    synthetic_preprocessed.shape[0], SEQUENCE_LENGTH, FEATURE_DIM
)

print(f"\nSynthetic log returns:")
print(f"  Range: [{synthetic_sequences.min():.4f}, {synthetic_sequences.max():.4f}]")
print(f"  Mean: {synthetic_sequences.mean():.6f}")
print(f"  Std: {synthetic_sequences.std():.6f}")

In [ ]:
# Visual comparison: real vs synthetic log returns
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Sample sequences
real_for_plot = log_returns_array[:SEQUENCE_LENGTH * 5].reshape(-1)
synth_for_plot = synthetic_sequences[:5].reshape(-1)

axes[0, 0].plot(real_for_plot, alpha=0.7, label='Real')
axes[0, 0].set_title('Real Log Returns (5 windows)')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(synth_for_plot, alpha=0.7, color='orange', label='Synthetic')
axes[0, 1].set_title('Synthetic Log Returns (5 windows)')
axes[0, 1].grid(True, alpha=0.3)

# Distribution comparison
axes[1, 0].hist(log_returns_array.ravel(), bins=100, density=True,
                alpha=0.6, label='Real')
axes[1, 0].hist(synthetic_sequences.ravel(), bins=100, density=True,
                alpha=0.6, label='Synthetic')
axes[1, 0].set_title('Distribution Comparison')
axes[1, 0].legend()
axes[1, 0].set_xlim(-0.05, 0.05)
axes[1, 0].grid(True, alpha=0.3)

# QQ-style: sorted real vs sorted synthetic
n_qq = min(5000, len(synthetic_sequences.ravel()))
real_sorted = np.sort(np.random.choice(log_returns_array.ravel(), n_qq, replace=False))
synth_sorted = np.sort(np.random.choice(synthetic_sequences.ravel(), n_qq, replace=False))
axes[1, 1].scatter(real_sorted, synth_sorted, s=1, alpha=0.3)
mn, mx = min(real_sorted.min(), synth_sorted.min()), max(real_sorted.max(), synth_sorted.max())
axes[1, 1].plot([mn, mx], [mn, mx], 'r--', alpha=0.5)
axes[1, 1].set_xlabel('Real quantiles')
axes[1, 1].set_ylabel('Synthetic quantiles')
axes[1, 1].set_title('QQ Plot')
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('QuantGAN: Real vs Synthetic', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Save Models & Synthetic Data

In [ ]:
# Save models
generator.save('../models/quantgan_generator.h5')
discriminator.save('../models/quantgan_discriminator.h5')

# Save synthetic data (log-return scale, for NB09 evaluation)
# Re-scale to [0,1] via MinMaxScaler to match NB09's expected format
from sklearn.preprocessing import MinMaxScaler

minmax_scaler = MinMaxScaler()
minmax_scaler.fit(log_returns_array)  # fit on real log returns

synthetic_minmax = minmax_scaler.transform(
    synthetic_sequences.reshape(-1, 1)
).reshape(synthetic_sequences.shape)

np.save('../data/processed/quantgan_synthetic.npy', synthetic_minmax)

# Also save raw log-return synthetic for direct analysis
np.save('../data/processed/quantgan_synthetic_logret.npy', synthetic_sequences)

print("Saved:")
print("  models/quantgan_generator.h5")
print("  models/quantgan_discriminator.h5")
print(f"  data/processed/quantgan_synthetic.npy — {synthetic_minmax.shape}")
print(f"  data/processed/quantgan_synthetic_logret.npy — {synthetic_sequences.shape}")

In [ ]:
# Quick summary statistics comparison
real_flat = log_returns_array.ravel()
synth_flat = synthetic_sequences.ravel()

print("="*60)
print("  QuantGAN — Quick Statistics")
print("="*60)
print(f"  {'Metric':<30} {'Real':>12} {'Synthetic':>12}")
print("-"*60)
print(f"  {'Mean':<30} {real_flat.mean():>12.6f} {synth_flat.mean():>12.6f}")
print(f"  {'Std':<30} {real_flat.std():>12.6f} {synth_flat.std():>12.6f}")
print(f"  {'Std ratio (synth/real)':<30} {'':>12} {synth_flat.std()/real_flat.std():>12.4f}")
print(f"  {'Skewness':<30} {float(pd.Series(real_flat).skew()):>12.4f} {float(pd.Series(synth_flat).skew()):>12.4f}")
print(f"  {'Kurtosis':<30} {float(pd.Series(real_flat).kurtosis()):>12.4f} {float(pd.Series(synth_flat).kurtosis()):>12.4f}")
print(f"  {'Min':<30} {real_flat.min():>12.6f} {synth_flat.min():>12.6f}")
print(f"  {'Max':<30} {real_flat.max():>12.6f} {synth_flat.max():>12.6f}")
print("="*60)